In [1]:
import time
import json
from typing import cast
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
import keras
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from visualize import save_confusion_matrix, save_class_metrics, save_class_distribution

In [2]:
BASE_DIR = Path.cwd()
DATA_DIR       = BASE_DIR / "data"
TRAIN_DIR      = DATA_DIR / "train"
VAL_DIR        = DATA_DIR / "val"
TEST_DIR       = DATA_DIR / "test"
TL_MODEL_PATH  = BASE_DIR / "weather_model_tl.h5"
TL_HISTORY_PNG = BASE_DIR / "training_history_tl.png"

In [3]:
IMG_SIZE    = (150, 150)
BATCH_SIZE  = 32
NUM_CLASSES = 5
SEED        = 42

In [4]:
CLASSES = ["cloudy", "foggy", "rainy", "shine", "sunrise"]

In [5]:
PHASE1_EPOCHS = 15
PHASE1_LR     = 1e-3

In [6]:
PHASE2_EPOCHS   = 15
PHASE2_LR       = 1e-5     
UNFREEZE_LAST_N = 30      

In [7]:
_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.10),
        layers.RandomZoom((-0.20, 0.20)),
        layers.RandomTranslation(height_factor=0.10, width_factor=0.10),
        layers.RandomBrightness(factor=0.15, value_range=(0, 1)),
        layers.RandomContrast(factor=0.15),
    ],
    name="augmentation",
)

In [8]:
def _to_float(images, labels):
    return tf.cast(images, tf.float32), labels

In [9]:
def _augment(images, labels):

    imgs_01 = images / 255.0
    imgs_01 = _augmentation(imgs_01, training=True)
    return imgs_01 * 255.0, labels

In [10]:
def _mobilenet_preprocess(images, labels):

    return mobilenet_preprocess(images), labels

In [11]:
def load_dataset(directory: Path, *, shuffle: bool, augment: bool) -> tf.data.Dataset:

    ds = cast(tf.data.Dataset, keras.utils.image_dataset_from_directory(
        str(directory),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        class_names=CLASSES,
        shuffle=shuffle,
        seed=SEED,
    ))
    ds = ds.map(_to_float, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_mobilenet_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [12]:
def compute_weights() -> dict:
    counts  = np.array(
        [len(list((TRAIN_DIR / cls).iterdir())) for cls in CLASSES], dtype=int
    )
    y_flat  = np.repeat(np.arange(NUM_CLASSES), counts)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y_flat,
    )
    weight_dict = {i: float(w) for i, w in enumerate(weights)}
    print("\nClass weights:")
    for i, cls in enumerate(CLASSES):
        print(f"  [{i}] {cls:<10}  weight={weight_dict[i]:.4f}  ({counts[i]} samples)")
    return weight_dict

In [13]:
def build_transfer_model():
  

    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )

    
    base_model.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="input")

    x = base_model(inputs, training=False)

    x = layers.Flatten(name="flatten")(x)

    x = layers.Dense(512, activation="relu", name="fc1")(x)
    x = layers.Dropout(0.50, name="dropout")(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)

    model = keras.Model(inputs, outputs, name="WeatherCNN_TL")
    return model, base_model

In [14]:
def unfreeze_top_n_layers(base_model: keras.Model, n: int) -> None:
  
    base_model.trainable = True  

    for layer in base_model.layers[:-n]:
        layer.trainable = False

    n_trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f"\n  Backbone: {n_trainable}/{len(base_model.layers)} layers unfrozen")

In [15]:
def save_history_plot(h1, h2) -> None:

    acc      = h1.history["accuracy"]     + h2.history["accuracy"]
    val_acc  = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss     = h1.history["loss"]         + h2.history["loss"]
    val_loss = h1.history["val_loss"]     + h2.history["val_loss"]
    p1_end   = len(h1.history["accuracy"])  # x-position of phase boundary

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(acc,    label="Train Accuracy")
    axes[0].plot(val_acc, label="Val Accuracy")
    axes[0].axvline(p1_end - 0.5, color="red", linestyle="--", label="Phase 2 start")
    axes[0].set_title("Transfer Learning — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(loss,     label="Train Loss")
    axes[1].plot(val_loss, label="Val Loss")
    axes[1].axvline(p1_end - 0.5, color="red", linestyle="--", label="Phase 2 start")
    axes[1].set_title("Transfer Learning — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(str(TL_HISTORY_PNG), dpi=150)
    plt.close(fig)
    print(f"History plot saved -> {TL_HISTORY_PNG}")

In [16]:
def print_comparison(tl_metrics: dict, phase1_params: int, phase2_params: int) -> None:

    scratch_path = BASE_DIR / "scratch_metrics.json"
    s = {}
    if scratch_path.exists():
        with open(scratch_path) as f:
            s = json.load(f)
    else:
        print("\n[INFO] scratch_metrics.json not found.")
        print("       Run train.py first to populate scratch CNN metrics.\n")

    def fmt_pct(val):
        return f"{val * 100:.2f}%" if val is not None else "N/A"

    def fmt_time(val):
        return f"{val:.0f}s" if val is not None else "N/A"

    def fmt_int(val):
        return f"{val:,}" if val is not None else "N/A"

    rows = [
        ("Metric",                     "Scratch CNN",                           "Transfer (MobileNetV2)"),
        ("-" * 35,                     "-" * 20,                                "-" * 22),
        ("Best Training Accuracy",     fmt_pct(s.get("best_train_accuracy")),   fmt_pct(tl_metrics["best_train_accuracy"])),
        ("Best Validation Accuracy",   fmt_pct(s.get("best_val_accuracy")),     fmt_pct(tl_metrics["best_val_accuracy"])),
        ("Final Test Accuracy",        fmt_pct(s.get("test_accuracy")),         fmt_pct(tl_metrics["test_accuracy"])),
        ("Total Training Time",        fmt_time(s.get("training_time_s")),      fmt_time(tl_metrics["training_time_s"])),
        ("Trainable Params (Phase 1)", fmt_int(s.get("trainable_params")),      fmt_int(phase1_params)),
        ("Trainable Params (Phase 2)", "N/A",                                   fmt_int(phase2_params)),
    ]

    print("\n" + "=" * 82)
    print("  MODEL COMPARISON: Scratch CNN vs. Transfer Learning (MobileNetV2)")
    print("=" * 82)
    for label, s_val, t_val in rows:
        print(f"  {label:<35} {s_val:<22} {t_val}")
    print("=" * 82)

In [17]:
if __name__ == "__main__":

    for d, name in [(TRAIN_DIR, "train"), (VAL_DIR, "val"), (TEST_DIR, "test")]:
        if not d.exists():
            raise FileNotFoundError(
                f"[ERROR] '{name}' directory not found: {d}\n"
                "Run prepare_dataset.py first."
            )

    print("Loading datasets (224x224 for MobileNetV2) ...")
    train_ds = load_dataset(TRAIN_DIR, shuffle=True,  augment=True)
    val_ds   = load_dataset(VAL_DIR,   shuffle=False, augment=False)
    test_ds  = load_dataset(TEST_DIR,  shuffle=False, augment=False)

    class_weights = compute_weights()

    print("\nBuilding transfer learning model ...")
    model, base_model = build_transfer_model()
    model.summary()

    print("\n" + "=" * 65)
    print("  PHASE 1: Training custom head (backbone fully frozen)")
    print("=" * 65)

    phase1_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    print(f"  Trainable parameters: {phase1_params:,}")
    print(f"  Learning rate: {PHASE1_LR}")
    print(f"  Max epochs: {PHASE1_EPOCHS}\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    phase1_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5,
            restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            str(TL_MODEL_PATH), monitor="val_accuracy",
            save_best_only=True, verbose=1,
        ),
    ]

    t_start = time.time()
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE1_EPOCHS,
        callbacks=phase1_callbacks,
        class_weight=class_weights,
    )


    print("\n" + "=" * 65)
    print(f"  PHASE 2: Fine-tuning last {UNFREEZE_LAST_N} backbone layers")
    print("=" * 65)

    unfreeze_top_n_layers(base_model, UNFREEZE_LAST_N)

    phase2_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    print(f"  Trainable parameters: {phase2_params:,}")
    print(f"  Learning rate: {PHASE2_LR}  (low LR protects pretrained weights)")
    print(f"  Max epochs: {PHASE2_EPOCHS}\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    phase2_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5,
            restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            str(TL_MODEL_PATH), monitor="val_accuracy",
            save_best_only=True, verbose=1,
        ),
    ]

    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE2_EPOCHS,
        callbacks=phase2_callbacks,
        class_weight=class_weights,
    )

    training_time = time.time() - t_start

    save_history_plot(h1, h2)

    print("\n" + "=" * 65)
    print("  FINAL EVALUATION ON HELD-OUT TEST SET")
    print("=" * 65)
    test_loss, test_acc = model.evaluate(test_ds)
    print(f"\n  Test Loss     : {test_loss:.4f}")
    print(f"  Test Accuracy : {test_acc * 100:.2f}%")

    all_train_acc = h1.history["accuracy"]     + h2.history["accuracy"]
    all_val_acc   = h1.history["val_accuracy"] + h2.history["val_accuracy"]

    tl_metrics = {
        "best_train_accuracy": float(max(all_train_acc)),
        "best_val_accuracy":   float(max(all_val_acc)),
        "test_accuracy":       float(test_acc),
        "test_loss":           float(test_loss),
        "training_time_s":     float(training_time),
        "phase1_params":       phase1_params,
        "phase2_params":       phase2_params,
    }
    with open(BASE_DIR / "tl_metrics.json", "w") as f:
        json.dump(tl_metrics, f, indent=2)

    print_comparison(tl_metrics, phase1_params, phase2_params)
    print(f"\nBest TL model saved -> {TL_MODEL_PATH}")

    save_class_distribution(TRAIN_DIR, VAL_DIR, TEST_DIR,
                            BASE_DIR / "class_distribution.png")
    y_true, y_pred = save_confusion_matrix(
        model, test_ds,
        BASE_DIR / "confusion_matrix_tl.png",
        title="Confusion Matrix - Transfer Learning / MobileNetV2 (Test Set)",
    )
    save_class_metrics(
        y_true, y_pred,
        BASE_DIR / "class_metrics_tl.png",
        title="Per-class Metrics - Transfer Learning / MobileNetV2 (Test Set)",
    )

Loading datasets (224x224 for MobileNetV2) ...
Found 1048 files belonging to 5 classes.
Found 226 files belonging to 5 classes.
Found 226 files belonging to 5 classes.

Class weights:
  [0] cloudy      weight=0.9981  (210 samples)
  [1] foggy       weight=0.9981  (210 samples)
  [2] rainy       weight=0.9981  (210 samples)
  [3] shine       weight=1.2046  (174 samples)
  [4] sunrise     weight=0.8590  (244 samples)

Building transfer learning model ...


C:\Users\h\AppData\Local\Temp\ipykernel_21284\1665820163.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "WeatherCNN_TL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 512)            │    16,384,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │         2,565 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,645,061 (71.13 MB)

...


Epoch 2: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 553ms/step - accuracy: 0.8015 - loss: 0.6357 - val_accuracy: 0.9071 - val_loss: 0.4294 - learning_rate: 1.0000e-05
Epoch 3/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.8506 - loss: 0.4575
Epoch 3: val_accuracy did not improve from 0.90708
33/33 ━━━━━━━━━━━━━━━━━━━━ 17s 490ms/step - accuracy: 0.8426 - loss: 0.4805 - val_accuracy: 0.9071 - val_loss: 0.4282 - learning_rate: 1.0000e-05
Epoch 4/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - accuracy: 0.8759 - loss: 0.4052
Epoch 4: val_accuracy improved from 0.90708 to 0.91150, saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5



Epoch 4: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 520ms/step - accuracy: 0.8712 - loss: 0.4076 - val_accuracy: 0.9115 - val_loss: 0.4343 - learning_rate: 1.0000e-05
Epoch 5/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - accuracy: 0.8437 - loss: 0.4403
Epoch 5: val_accuracy improved from 0.91150 to 0.92035, saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5



Epoch 5: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 561ms/step - accuracy: 0.8645 - loss: 0.3868 - val_accuracy: 0.9204 - val_loss: 0.4384 - learning_rate: 1.0000e-05
Epoch 6/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.8711 - loss: 0.3809
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.

Epoch 6: val_accuracy did not improve from 0.92035
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 522ms/step - accuracy: 0.8740 - loss: 0.3456 - val_accuracy: 0.9159 - val_loss: 0.4328 - learning_rate: 1.0000e-05
Epoch 7/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.8654 - loss: 0.4128
Epoch 7: val_accuracy improved from 0.92035 to 0.92478, saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5



Epoch 7: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 557ms/step - accuracy: 0.8578 - loss: 0.4550 - val_accuracy: 0.9248 - val_loss: 0.4196 - learning_rate: 5.0000e-06
Epoch 8/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.8795 - loss: 0.3440
Epoch 8: val_accuracy did not improve from 0.92478
33/33 ━━━━━━━━━━━━━━━━━━━━ 17s 493ms/step - accuracy: 0.8845 - loss: 0.3325 - val_accuracy: 0.9248 - val_loss: 0.4155 - learning_rate: 5.0000e-06
Epoch 9/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - accuracy: 0.8706 - loss: 0.3777
Epoch 9: val_accuracy improved from 0.92478 to 0.92920, saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5



Epoch 9: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model_tl.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 549ms/step - accuracy: 0.8750 - loss: 0.3604 - val_accuracy: 0.9292 - val_loss: 0.4147 - learning_rate: 5.0000e-06
Epoch 10/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.8798 - loss: 0.3644
Epoch 10: val_accuracy did not improve from 0.92920
33/33 ━━━━━━━━━━━━━━━━━━━━ 17s 478ms/step - accuracy: 0.8683 - loss: 0.3728 - val_accuracy: 0.9292 - val_loss: 0.4136 - learning_rate: 5.0000e-06
Epoch 11/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.8786 - loss: 0.3331
Epoch 11: val_accuracy did not improve from 0.92920
33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 483ms/step - accuracy: 0.8836 - loss: 0.3210 - val_accuracy: 0.9248 - val_loss: 0.4111 - learning_rate: 5.0000e-06
Epoch 12/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.8741 - loss: 0.3722
Epoch 12: val_accuracy did not improve from 0.92920
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 506ms/step - accuracy: 0.88